# 04 — Analysis & Visualization
Explore final results, generate interactive BERTopic visualizations, and export classified datasets.

In [ ]:
import sys
sys.path.insert(0, '../src')

import yaml
from pathlib import Path

from reddit_np_topics.db import get_connection, read_table, list_parks
from reddit_np_topics.modeling.train_global import load_model
from reddit_np_topics.visualization.plots import plot_all_global, plot_regional

with open('../config.yaml') as f:
    cfg = yaml.safe_load(f)

conn = get_connection(Path('..') / cfg['paths']['database'])
vis_dir = Path('..') / cfg['paths']['outputs_visualizations']
results_dir = Path('..') / cfg['paths']['outputs_results']

In [ ]:
# Load final classified data
df = read_table(conn, 'regional_classified')
print(f'{len(df)} documents, {df.park_name.nunique()} parks')
df.head(3)

In [ ]:
# Load global model
global_model = load_model(
    Path('..') / cfg['paths']['models_global'] / 'global_model',
    cfg['modeling']['embedding_model'],
)

In [ ]:
# Generate all global visualizations
plot_all_global(
    model=global_model,
    docs=df['tokens'].tolist(),
    park_names=df['park_name'].tolist(),
    output_dir=vis_dir,
    top_n_topics=36,
)

In [ ]:
# Topic distribution overview
global_model.get_topic_info()

In [ ]:
# Topics per park — what does each park talk about?
df.groupby(['park_name', 'Global_Topic']).size().unstack(fill_value=0)

In [ ]:
# Export final results to CSV
results_dir.mkdir(parents=True, exist_ok=True)
df.to_csv(results_dir / 'final_classified_all_parks.csv', index=False, encoding='utf-8')
print('Results saved.')